# Formatos de arquivos e exportação para SQL
Leitura e escrita de dados com pandas em diferentes formatos (CSV, Excel, Google Sheets, JSON, HTML, XML e Parquet), detecção de *encoding*, uso de bancos de dados com SQLAlchemy e geração de scripts SQL `INSERT` a partir de DataFrames, incluindo a normalização de uma tabela de livros em autores, gêneros, livros e comentários.

**Origem:** cursos Alura (pandas: formatos de arquivos; engenharia de dados: limpeza e exportação SQL) e estudos próprios (Parquet).

**Dados:**
- Por URL (baixados direto): `superstore_data.csv`, `emissoes_CO2.xlsx`, `clientes_banco.csv` (repositório `alura-cursos/Pandas`), planilha pública do Google Sheets, API `jsonplaceholder.typicode.com` e `comentarios.json` (repositório `guilhermeonrails/datas-csv`).
- Enviar para `/content/` no Colab: `aluguel.csv`, `pacientes.json`, `filmes_wikipedia.html`, `imdb_top_1000.xml`, `livros.csv` e os arquivos Parquet `answers`, `choices`, `evaluations`, `questions` e `stores` (`.parquet`).

**Índice**
1. CSV e *encodings*
2. Excel e Google Sheets
3. JSON
4. HTML e XML
5. Parquet
6. Bancos de dados com SQLAlchemy
7. Gerando scripts SQL a partir de DataFrames
8. Normalização de livros e exportação SQL

In [ ]:
# Setup: deep-translator não vem instalado no Colab
!pip install -q deep-translator

import chardet
import pandas as pd
import pyarrow.parquet as pq
import requests
from deep_translator import GoogleTranslator
from sqlalchemy import create_engine, inspect, text

## 1. CSV e *encodings*
Antes de ler um arquivo de origem desconhecida, o `chardet` estima a codificação (*encoding*) a partir dos bytes do arquivo; o resultado pode ser passado em `pd.read_csv(..., encoding=...)`. Na leitura, `nrows` limita a quantidade de linhas e `usecols` seleciona colunas; na escrita, `sep` define o separador e `index=False` evita gravar o índice.

In [ ]:
with open('/content/aluguel.csv', 'rb') as arquivo:
    print(chardet.detect(arquivo.read()))

In [ ]:
url = 'https://raw.githubusercontent.com/alura-cursos/pandas/refs/heads/main/superstore_data.csv'
df = pd.read_csv(url, nrows=7, sep=',', usecols=['Id', 'Year_Birth', 'Income'])
df

In [ ]:
df.to_csv('superstore_formatado.csv', index=False, sep=';')

## 2. Excel e Google Sheets
`pd.ExcelFile(...).sheet_names` lista as abas de uma planilha; `sheet_name` escolhe a aba e `usecols='A:D'` aceita intervalos no estilo do Excel.

In [ ]:
url2 = 'https://github.com/alura-cursos/Pandas/blob/main/emissoes_CO2.xlsx?raw=True'
print(pd.ExcelFile(url2).sheet_names)  # lê todas as abas/páginas do Excel e as retorna

df_emissoes_percapita = pd.read_excel(url2, sheet_name='emissoes_percapita')
df_fontes = pd.read_excel(url2, sheet_name='fontes')
df_emissoes_co2 = pd.read_excel(url2, sheet_name='emissoes_C02', usecols='A:D', nrows=10)

In [ ]:
df_emissoes_co2.to_excel('emissoes_co2.xlsx', index=False)
emissoes = pd.read_excel('/content/emissoes_co2.xlsx')
emissoes

Uma planilha pública do Google Sheets pode ser lida como CSV pela URL `gviz/tq?tqx=out:csv&sheet=<aba>`.

In [ ]:
sheet_id = '1lzq0k-41-MbbS63C3Q9i1wPvLkSJt9zhr4Jolt1vEog'


def ler_aba_sheets(sheet_id, aba):
    url_aba = f'https://docs.google.com/spreadsheets/d/{sheet_id}/gviz/tq?tqx=out:csv&sheet={aba}'
    return pd.read_csv(url_aba)


fontes_sheets = ler_aba_sheets(sheet_id, 'fontes')
emissoes_sheets = ler_aba_sheets(sheet_id, 'emissoes_percapita')
emissoes_sheets.head()

## 3. JSON
Dados JSON "planos" são lidos direto com `pd.read_json`. Dados aninhados precisam de `pd.json_normalize`, que transforma dicionários internos em colunas (`Saude.IMC`, por exemplo); com `record_path` e `meta` é possível expandir uma lista interna mantendo campos do nível superior.

In [ ]:
# Dados não aninhados
dados_pacientes = pd.read_json('/content/pacientes.json')
dados_pacientes.head()

In [ ]:
# JSON em uma LISTA
json_lista = [
    {'ID': '01', 'Faixa_etaria': '55-59', 'Sexo_biologico': 'feminino'},
    {'ID': '02', 'Faixa_etaria': '80 ou +', 'Sexo_biologico': 'feminino'}
]
pd.json_normalize(json_lista)

In [ ]:
# JSON em uma LISTA ANINHADA
json_list = [
    {
        'ID': '01',
        'Faixa_etaria': '55-59',
        'Sexo_biologico': 'Feminino',
        'Saude': {'Dificuldade_caminhar': 'Nao',
                  'Atividade_fisica': 'Sim',
                  'IMC': 16.6,
                  'Doenca_cardiaca': 'Nao'}
    },
    {
        'ID': '02',
        'Faixa_etaria': '80 ou +',
        'Sexo_biologico': 'Feminino',
        'Saude': {'Dificuldade_caminhar': 'Nao',
                  'Atividade_fisica': 'Sim',
                  'IMC': 20.34,
                  'Doenca_cardiaca': 'Sim'}
    }
]
json_lista_normalizado = pd.json_normalize(json_list)
json_lista_normalizado

In [ ]:
# JSON em um DICIONÁRIO com uma lista de registros
dados_dict = {
    "Pesquisa": "Principais Indicadores de Doenca Cardiaca",
    "Ano": 2020,
    "Pacientes": [
        {
            "ID": "01",
            "Faixa_etaria": "55-59",
            "Sexo_biologico": "Feminino",
            "Raça": "Branca",
            "IMC": 16.6,
            "Fumante": "Sim",
            "Consumo_alcool": "Nao",
            "Saude_fisica": 3,
            "Saude_mental": 30,
            "Dificuldade_caminhar": "Nao",
            "Atividade_fisica": "Sim",
            "Saude_geral": "Muito boa",
            "Horas_sono": 5,
            "Problemas_saude": ["Diabetes", "Asma", "Cancer_pele"]
        },
        {
            "ID": "02",
            "Faixa_etaria": "80 ou +",
            "Sexo_biologico": "Feminino",
            "Raça": "Branca",
            "IMC": 20.34,
            "Fumante": "Nao",
            "Consumo_alcool": "Nao",
            "Saude_fisica": 0,
            "Saude_mental": 0,
            "Dificuldade_caminhar": "Nao",
            "Atividade_fisica": "Sim",
            "Saude_geral": "Muito boa",
            "Horas_sono": 7,
            "Problemas_saude": ["AVC"]
        },
        {
            "ID": "03",
            "Faixa_etaria": "65-69",
            "Sexo_biologico": "Masculino",
            "Raça": "Branca",
            "IMC": 26.58,
            "Fumante": "Sim",
            "Consumo_alcool": "Nao",
            "Saude_fisica": 20,
            "Saude_mental": 30,
            "Dificuldade_caminhar": "Nao",
            "Atividade_fisica": "Sim",
            "Saude_geral": "Muito boa",
            "Horas_sono": 8,
            "Problemas_saude": ["diabetes", "Asma"]
        }
    ]
}

json_dicionario_normalizado = pd.json_normalize(dados_dict, record_path=['Pacientes'], meta=['Pesquisa', 'Ano'])
json_dicionario_normalizado

In [ ]:
# Escrita e leitura de volta
json_dicionario_normalizado.to_json('Pacientes.json')
dfJson = pd.read_json('/content/Pacientes.json')
dfJson

### 3.1 JSON vindo de uma API
A resposta da API é uma lista de usuários com campos aninhados (`address`, `company`). Por padrão o `json_normalize` separa os níveis com `.`; `sep='_'` troca o separador.

In [ ]:
url = 'https://jsonplaceholder.typicode.com/users'
resultado = requests.get(url).json()

resultado_normalizado = pd.json_normalize(resultado)
resultado_normalizado2 = pd.json_normalize(resultado, sep='_')
resultado_normalizado2.head()

## 4. HTML e XML
`pd.read_html` devolve uma **lista** com todas as tabelas encontradas na página; escolhe-se a tabela pelo índice. `pd.read_xml` lê registros repetidos de um XML.

In [ ]:
tabelas_html = pd.read_html('/content/filmes_wikipedia.html')
print(len(tabelas_html), 'tabelas encontradas')
top_filmes = tabelas_html[1]
top_filmes

In [ ]:
top_filmes.to_html('top_filmes.html')
top_filmes.to_csv('top_filmes_1998.csv', index=False)

In [ ]:
dados_imdb = pd.read_xml('/content/imdb_top_1000.xml')
dados_imdb.head()

In [ ]:
dados_imdb.to_xml('filmes_imdb.xml')

## 5. Parquet
Parquet é um formato colunar e comprimido, comum em engenharia de dados. O `pyarrow` lê o arquivo como uma `Table` (`pq.read_table`), que é convertida para pandas com `.to_pandas()`; `pd.read_parquet` faz as duas etapas de uma vez. Aqui as cinco tabelas são carregadas e convertidas para CSV.

In [ ]:
nomes_parquet = ['answers', 'choices', 'evaluations', 'questions', 'stores']
tabelas_parquet = {nome: pq.read_table(f'{nome}.parquet').to_pandas() for nome in nomes_parquet}

for nome, tabela in tabelas_parquet.items():
    tabela.to_csv(f'{nome}.csv', index=False)
    print(nome, list(tabela.columns))

In [ ]:
# Equivalente direto com pandas
answers = pd.read_parquet('answers.parquet')
answers.head()

## 6. Bancos de dados com SQLAlchemy
- `create_engine` cria o "motor" que faz a conexão com o banco (aqui um SQLite em memória).
- `inspect` inspeciona a estrutura do banco (tabelas, colunas).
- `to_sql` grava um DataFrame como tabela; `read_sql` lê o resultado de uma consulta e `read_sql_table` lê uma tabela completa.

In [ ]:
engine = create_engine('sqlite:///:memory:')

url = 'https://raw.githubusercontent.com/alura-cursos/Pandas/refs/heads/main/clientes_banco.csv'
clientes = pd.read_csv(url)
clientes.to_sql('clientes', engine, index=False)

inspect(engine).get_table_names()

In [ ]:
query = "SELECT * FROM clientes WHERE Categoria_de_renda = 'Empregado'"
empregados = pd.read_sql(query, engine)
empregados.to_sql('empregados', con=engine, index=False)

pd.read_sql_table('empregados', engine, columns=['ID_Cliente', 'Grau_escolaridade', 'Rendimento_anual'])

### 6.1 Alterando dados (UPDATE, DELETE, INSERT)
No SQLAlchemy 2.x, comandos em texto precisam ser envolvidos por `text()` e confirmados com `conn.commit()`.

In [ ]:
comandos = [
    "UPDATE clientes SET Grau_escolaridade = 'Ensino superior' WHERE ID_Cliente = 5008808",
    "UPDATE clientes SET Rendimento_anual = 300000.0 WHERE ID_Cliente = 6840104",
    "DELETE FROM clientes WHERE ID_Cliente = 5008804",
    "DELETE FROM clientes WHERE ID_Cliente = 5008809",
    "INSERT INTO clientes (ID_Cliente, Idade, Grau_escolaridade, Estado_civil, "
    "Tamanho_familia, Categoria_de_renda, Ocupacao, Anos_empregado, "
    "Rendimento_anual, Tem_carro, Moradia) "
    "VALUES (6850985, 33, 'Doutorado', 'Solteiro', 1, 'Empregado', 'TI', "
    "2, 290000, 0, 'Casa/apartamento próprio')",
]

with engine.connect() as conn:
    for comando in comandos:
        conn.execute(text(comando))
    conn.commit()

pd.read_sql_table('clientes', engine).tail()

## 7. Gerando scripts SQL a partir de DataFrames
Para carregar dados em um banco externo, é comum gerar um arquivo `.sql` com comandos `INSERT`. Duas regras garantem um SQL válido: valores nulos viram `NULL` e aspas simples dentro de textos são escapadas (`'` → `''`). A função `gerar_insert_sql` aplica essas regras a qualquer DataFrame e substitui os blocos repetidos de `INSERT` (um por tabela) dos notebooks originais.

In [ ]:
def formatar_valor(valor):
    """Converte um valor Python/pandas para literal SQL."""
    if pd.isna(valor):
        return 'NULL'
    if isinstance(valor, pd.Timestamp):
        return f"'{valor}'"
    if isinstance(valor, str):
        valor = valor.replace("'", "''")  # escapa aspas simples para SQL
        return f"'{valor}'"
    return str(valor)


def gerar_insert_sql(df, tabela, arquivo, colunas=None, multi_linhas=False):
    """Grava em `arquivo` os comandos INSERT de `df` na `tabela`.

    colunas: colunas a inserir (padrão: todas, na ordem do DataFrame).
    multi_linhas: se True, gera um único INSERT com várias tuplas em VALUES.
    """
    colunas = list(df.columns) if colunas is None else list(colunas)
    rotulos = ', '.join(colunas)
    tuplas = [
        '(' + ', '.join(formatar_valor(v) for v in linha) + ')'
        for linha in df[colunas].itertuples(index=False, name=None)
    ]
    with open(arquivo, 'w', encoding='utf-8') as f:
        if multi_linhas and tuplas:
            f.write(f'INSERT INTO {tabela} ({rotulos}) VALUES\n' + ',\n'.join(tuplas) + ';\n')
        else:
            for tupla in tuplas:
                f.write(f'INSERT INTO {tabela} ({rotulos}) VALUES {tupla};\n')
    print(f"Arquivo '{arquivo}' gerado com {len(tuplas)} registro(s).")


# A mesma função serve para qualquer CSV, por exemplo:
# gerar_insert_sql(pd.read_csv('alunos.csv'), 'alunos', 'alunos.sql')

### 7.1 Exportando tabelas do Parquet
As tabelas `choices` e `evaluations` (seção 5) são exportadas com as suas três primeiras colunas; a segunda usa o formato de um único `INSERT` com várias linhas.

In [ ]:
choices = tabelas_parquet['choices']
evaluations = tabelas_parquet['evaluations']

gerar_insert_sql(choices, 'choices', 'choices.sql', colunas=choices.columns[:3])
gerar_insert_sql(evaluations, 'evaluations', 'evaluations.sql', colunas=evaluations.columns[:3], multi_linhas=True)

## 8. Normalização de livros e exportação SQL
A tabela `livros.csv` (livros mais vendidos) tem autor e gênero repetidos em cada linha. Para montar um banco relacional, ela é dividida em quatro tabelas: `autores`, `generos`, `livros` (com as chaves `autor_id` e `genero_id`) e `comentarios` (com `livro_id`), cujos dados vêm de uma API JSON.

### 8.1 Carga e renomeação das colunas

In [ ]:
df = pd.read_csv('/content/livros.csv')

# renomear as colunas para o português
traducao = {
    'Book': 'livro',
    'Author(s)': 'autor',
    'Original language': 'idioma_original',
    'First published': 'ano_publicacao',
    'Approximate sales in millions': 'vendas',
    'Genre': 'genero'
}
df.rename(columns=traducao, inplace=True)
print(df.shape)
df.info()

In [ ]:
# dados nulos e autores com mais livros na lista
print(df.isnull().sum())
df['autor'].value_counts().head(10)

### 8.2 Tabelas de autores e gêneros (chaves)
`.unique()` retorna cada autor uma única vez; o índice + 1 vira um identificador sequencial (como um `AUTOINCREMENT`). O `merge` com `how='left'` mantém todas as linhas de `df` e acrescenta o `autor_id` correspondente. O mesmo é feito para os gêneros, depois de preencher os gêneros ausentes com `'Unknown'`.

In [ ]:
autores_unicos = pd.DataFrame(df['autor'].unique(), columns=['autor'])
autores_unicos['autor_id'] = autores_unicos.index + 1
df = df.merge(autores_unicos, on='autor', how='left')

df['genero'] = df['genero'].fillna('Unknown')
generos_unicos = pd.DataFrame(df['genero'].unique(), columns=['genero'])
generos_unicos['genero_id'] = generos_unicos.index + 1
df = df.merge(generos_unicos, on='genero', how='left')

df[df['autor'] == 'J. K. Rowling']

Os nomes dos gêneros são traduzidos para o português com o `deep_translator` (usa o Google Tradutor, precisa de internet) e ajustados manualmente.

In [ ]:
generos_unicos['genero_pt'] = generos_unicos['genero'].apply(
    lambda x: GoogleTranslator(source='auto', target='pt').translate(x)
)
generos_unicos['genero_pt'] = generos_unicos['genero_pt'].replace('Novella', 'Novela').str.lower()
generos_unicos.head()

### 8.3 Comentários (API JSON)
Cada comentário traz o título do livro; o `livro_id` vem da posição do livro em `df` (índice + 1, a mesma ordem em que os livros serão inseridos).

In [ ]:
api = 'https://raw.githubusercontent.com/guilhermeonrails/datas-csv/refs/heads/main/comentarios.json'
df_comentarios = pd.read_json(api)
print(df_comentarios.shape)

df_comentarios = df_comentarios.merge(
    df[['livro']].reset_index().rename(columns={'index': 'livro_id'}),
    on='livro',
    how='left'
)
df_comentarios['livro_id'] = (df_comentarios['livro_id'] + 1).astype('Int64')

print(df_comentarios.isnull().sum())
print(f"livro_id: mínimo {df_comentarios['livro_id'].min()}, máximo {df_comentarios['livro_id'].max()}")

### 8.4 Geração dos scripts SQL
Os IDs não são gravados em `autores` e `generos`: a ordem das linhas é a mesma usada para gerar `autor_id` e `genero_id`, então o `AUTOINCREMENT` do banco reproduz as chaves.

In [ ]:
autores = autores_unicos.rename(columns={'autor': 'nome'})
gerar_insert_sql(autores, 'autores', 'autores.sql', colunas=['nome'])

generos = generos_unicos.rename(columns={'genero_pt': 'nome'})
gerar_insert_sql(generos, 'generos', 'generos.sql', colunas=['nome'])

livros = df.rename(columns={'livro': 'nome', 'idioma_original': 'idioma'})
livros = livros.astype({'ano_publicacao': int, 'autor_id': int, 'genero_id': int})
livros['vendas'] = livros['vendas'].astype(float).round(2)
gerar_insert_sql(livros, 'livros', 'livros.sql',
                 colunas=['nome', 'idioma', 'ano_publicacao', 'vendas', 'autor_id', 'genero_id'])

gerar_insert_sql(df_comentarios, 'comentarios', 'comentarios.sql',
                 colunas=['livro_id', 'nome', 'sobrenome', 'comentario'])